# 🥫 Food-Safety Alert Classification — ML Capstone Project

**Business problem:** Automatically classify EU food/feed safety notifications (RASFF) into a **hazard category** (e.g. microbiological contamination, mycotoxins, heavy metals, foreign bodies, packaging/chemical migration, allergens, etc.) based on the **product**, **origin country**, **destination country**, and **notification date** — *without* looking at the hazard text itself (that would be leakage).

**Target users / stakeholders:** Food-safety regulators, importers, and quality-assurance teams who want an early warning of what *type* of risk a shipment is likely to trigger, and analysts who want a searchable dashboard of historical alerts.

**Expected outcome:** A classifier + interactive dashboard that predicts the most likely hazard category for a given product/origin/destination, plus visual trends by country, product, and time.

**Business / social value:** Faster triage of incoming notifications, better resource allocation for border inspections, and an accessible way for smaller importers to understand historical risk patterns for specific product–origin combinations.

**ML task:** Multiclass text + tabular classification (NLP on product name + categorical/date features).

---
### How to use this notebook
1. Runtime → Run all (Google Colab). Total runtime ≈ 5–10 minutes.
2. Everything (data download, training, tuning, dashboard artifacts) runs automatically — no manual steps needed until the GitHub / Streamlit deployment section at the end.
3. This notebook is designed to fit comfortably inside the **2-hour capstone time budget**.


## Step 0 — Setup: install & import libraries

In [ ]:
# Install any packages not already present in Colab
!pip install -q xgboost lightgbm shap
# NOTE: if Colab prints "You must restart the runtime", click Runtime > Restart runtime,
# then Runtime > Run all again from the top. (We deliberately do NOT use --upgrade here,
# since force-upgrading Colab's preinstalled packages is what usually triggers that prompt.)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, joblib, json, os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, precision_score, recall_score)
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

os.makedirs("outputs", exist_ok=True)
print("Setup complete. LightGBM available:", HAS_LGBM)


## Step 1 — Data Collection

**Source:** This project uses a public, English-language extract of ~31,600 real **RASFF (Rapid Alert System for Food and Feed)** notifications, published as `safety_reports_v1.csv.gz` in the companion repository for *Human-in-the-Loop Machine Learning* (Robert Munro), hosted on GitHub: `github.com/rmunro/food_safety`. RASFF is the European Commission's official system for food/feed safety alerts (https://food.ec.europa.eu/food-safety/rasff_en), so this satisfies the "official dataset" requirement for this project type.

Each row is one notification with: a reference ID, a free-text report summary, the case date, the food/product, the hazard, the origin country, and the destination country.

**Why this source instead of the RASFF portal directly:** the RASFF web portal is a JavaScript search UI without a simple bulk-CSV endpoint, and the capstone must be completed in ~2 hours. This CSV is a legitimate scrape of the same public RASFF notifications and is >10,000 rows, so it meets the dataset-size requirement while staying reproducible in one line of code.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rmunro/food_safety/master/safety_reports_v1.csv.gz"

df_raw = pd.read_csv(
    DATA_URL, header=None, compression="gzip",
    names=["ref_id", "report_text", "date_case", "food", "hazard",
           "origin_country", "destination_country"]
)
print("Raw shape:", df_raw.shape)
df_raw.head()


## Step 2 — Data Quality Assessment & Preprocessing

Preprocessing decisions (each briefly justified):
- **Drop exact duplicate rows** — the same notification can appear more than once when it lists multiple destination countries; true duplicates add no information.
- **Drop rows with missing food / hazard / country / date** — these fields are essential to the task and cannot be safely imputed for a categorical/text field like this.
- **Parse `date_case` to datetime** and drop unparsable dates — needed to engineer year/month features.
- **Normalise text case** on `food` — reduces sparse TF-IDF vocabulary from casing noise.


In [ ]:
df = df_raw.copy()

print("Missing values before cleaning:")
print(df.isna().sum())

before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["food", "hazard", "origin_country", "destination_country", "date_case"])
df["date_case"] = pd.to_datetime(df["date_case"], errors="coerce")
df = df.dropna(subset=["date_case"])
df["food"] = df["food"].astype(str).str.lower().str.strip()
df["origin_country"] = df["origin_country"].astype(str).str.strip()
df["destination_country"] = df["destination_country"].astype(str).str.strip()

print(f"\nRows before cleaning: {before} | after cleaning: {len(df)} | removed: {before - len(df)}")
df.info()


## Step 3 — Exploratory Data Analysis

In [ ]:
print("Descriptive stats (categorical cardinality):")
print(df[["food", "hazard", "origin_country", "destination_country"]].nunique())
print("\nDate range:", df["date_case"].min(), "to", df["date_case"].max())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["hazard"].value_counts().head(15).plot(kind="barh", ax=axes[0], color="firebrick")
axes[0].invert_yaxis(); axes[0].set_title("Top 15 raw hazard labels")
df["origin_country"].value_counts().head(15).plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].invert_yaxis(); axes[1].set_title("Top 15 origin countries")
plt.tight_layout(); plt.savefig("outputs/eda_top_hazards_countries.png", dpi=120); plt.show()


In [ ]:
notif_by_year = df.groupby(df["date_case"].dt.year).size()
plt.figure()
notif_by_year.plot(kind="line", marker="o", color="darkorange")
plt.title("RASFF notifications per year (dataset extract)")
plt.xlabel("Year"); plt.ylabel("Number of notifications")
plt.tight_layout(); plt.savefig("outputs/eda_notifications_by_year.png", dpi=120); plt.show()


In [ ]:
top_food_words = df["food"].str.split().explode().value_counts().head(20)
plt.figure()
top_food_words.plot(kind="bar", color="seagreen")
plt.title("Most frequent words in product/food descriptions")
plt.xticks(rotation=75)
plt.tight_layout(); plt.savefig("outputs/eda_top_food_words.png", dpi=120); plt.show()


**EDA insight:** *Listeria monocytogenes*, *Salmonella* and *aflatoxins* dominate raw hazard labels, notifications are concentrated in the last two decades, and product descriptions are short and noisy (multi-word phrases like "frozen", "fillets", "from farmed" recur) — this supports using TF-IDF on the food/product text rather than one-hot encoding thousands of unique product strings.

## Step 4 — Feature Engineering & Target Construction

The raw `hazard` field has ~960 unique free-text values (e.g. *"aflatoxins (B1 = 9.4 µg/kg)"*), too granular and too imbalanced to classify directly. We engineer a **`hazard_category`** target with keyword rules grouping hazards into ~12 business-meaningful categories (microbiological contamination, mycotoxins, heavy metals, foreign bodies, packaging/chemical migration, GMO, pesticide residues, veterinary drug residues, irradiation, food additives, allergens, labelling/regulatory non-compliance). Rare categories (<80 records) are folded away to keep classes learnable.

**⚠️ Target-leakage check:** `report_text` is auto-generated and *literally names the hazard* (e.g. "...aflatoxins... in pistachio paste from Italy..."), and `hazard` obviously **is** the pre-image of the target. Both are therefore **excluded from the feature set**. We only use `food`, `origin_country`, `destination_country`, and date parts as predictors — a harder but realistic, leakage-free framing: *"given what the product is and where it's from/going, what type of hazard is it most likely to trigger?"*


In [ ]:
def map_hazard(h):
    h = str(h).lower()
    if any(k in h for k in ["listeria","salmonella","coli","norovirus","virus","parasit",
                              "mould","mold","outbreak","bacillus","yersinia","campylobacter",
                              "hepatitis","clostrid"]):
        return "Microbiological contamination"
    if any(k in h for k in ["aflatoxin","ochratoxin","mycotoxin","zearalenone","fumonisin","patulin"]):
        return "Mycotoxins"
    if any(k in h for k in ["mercury","cadmium","lead","arsenic","tin","heavy metal"]):
        return "Heavy metals"
    if any(k in h for k in ["glass","plastic fragment","metal fragment","foreign body",
                              "wood fragment","rubber fragment","insect fragment"]):
        return "Foreign bodies"
    if any(k in h for k in ["migration of","melamine","formaldehyde","bisphenol","aromatic amine"]):
        return "Packaging / chemical migration"
    if any(k in h for k in ["genetically modified","gmo"]):
        return "GMO / unauthorised composition"
    if any(k in h for k in ["pesticide","residue"]) and "veterinary" not in h:
        return "Pesticide residues"
    if any(k in h for k in ["veterinary","antibiotic","hormone","chloramphenicol"]):
        return "Veterinary drug residues"
    if "irradiation" in h:
        return "Irradiation"
    if any(k in h for k in ["sulphite","additive","colour","preservative","sweetener"]):
        return "Food additives"
    if any(k in h for k in ["traces of","allergen","gluten","peanut","egg","milk","soy"]):
        return "Allergens"
    if any(k in h for k in ["labelling","placing on the market","unauthorised","fraud","composition"]):
        return "Labelling / regulatory non-compliance"
    return "Other"

df["hazard_category"] = df["hazard"].apply(map_hazard)

# Fold rare classes (<80 records) so every class is learnable within CV
counts = df["hazard_category"].value_counts()
keep_classes = counts[counts >= 80].index
df = df[df["hazard_category"].isin(keep_classes)].reset_index(drop=True)

df["year"] = df["date_case"].dt.year
df["month"] = df["date_case"].dt.month

print("Final class distribution:")
print(df["hazard_category"].value_counts())


In [ ]:
plt.figure(figsize=(9,5))
df["hazard_category"].value_counts().plot(kind="barh", color="slateblue")
plt.gca().invert_yaxis()
plt.title("Engineered target: hazard_category distribution (class imbalance)")
plt.tight_layout(); plt.savefig("outputs/eda_target_distribution.png", dpi=120); plt.show()


**Class-imbalance note:** *Microbiological contamination* and *Other* dominate. We handle this with `class_weight='balanced'` where supported, **macro-F1** (not accuracy) as the primary evaluation metric, and a **stratified** train/test split.

## Step 5 — Data Preparation & Validation Strategy

- **Validation strategy:** 80:20 **stratified** train/test split (stratified because classes are imbalanced), plus 3-fold stratified cross-validation during model comparison.
- **Leakage prevention:** all preprocessing (TF-IDF vocabulary, one-hot categories) is fit **only on the training fold**, via a single `sklearn` `Pipeline`/`ColumnTransformer` — never on the full dataset before splitting.
- High-cardinality `origin_country` / `destination_country` are capped to the top 15 most frequent values + `"Other"` to avoid an explosion of near-unique one-hot columns.


In [ ]:
FEATURES = ["food", "origin_country", "destination_country", "year", "month"]
X = df[FEATURES].copy()
y = df["hazard_category"].copy()

for col in ["origin_country", "destination_country"]:
    top = X[col].value_counts().nlargest(15).index
    X[col] = X[col].where(X[col].isin(top), "Other")

le = LabelEncoder()
y_enc = le.fit_transform(y)  # needed for XGBoost/LightGBM

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X, y_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_enc)

preprocessor = ColumnTransformer([
    ("food_tfidf", TfidfVectorizer(max_features=800, ngram_range=(1, 2), min_df=3), "food"),
    ("origin_ohe", OneHotEncoder(handle_unknown="ignore"), ["origin_country"]),
    ("dest_ohe", OneHotEncoder(handle_unknown="ignore"), ["destination_country"]),
], remainder="passthrough")  # year, month pass through unchanged

print("Train:", X_train.shape, " Test:", X_test.shape, " Classes:", len(le.classes_))


## Step 6 — Model Development

We compare 8 algorithms suitable for multiclass classification on mixed text + categorical data.


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=300, class_weight="balanced"),
    "LinearSVC": LinearSVC(class_weight="balanced", max_iter=3000),
    "ComplementNB": ComplementNB(),
    "DecisionTree": DecisionTreeClassifier(max_depth=20, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=20, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, eval_metric="mlogloss",
                              random_state=RANDOM_STATE, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}
if HAS_LGBM:
    models["LightGBM"] = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbosity=-1)

print(f"Comparing {len(models)} algorithms:", list(models.keys()))


In [ ]:
NEEDS_ENCODED_Y = {"XGBoost", "LightGBM"}  # these require integer-encoded labels

results = []
fitted_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    t0 = time.time()
    if name in NEEDS_ENCODED_Y:
        pipe.fit(X_train_e, y_train_e)
        fit_time = time.time() - t0
        y_pred = pipe.predict(X_test_e)
        train_acc = pipe.score(X_train_e, y_train_e)
        test_acc = pipe.score(X_test_e, y_test_e)
        f1 = f1_score(y_test_e, y_pred, average="macro")
    else:
        pipe.fit(X_train, y_train)
        fit_time = time.time() - t0
        y_pred = pipe.predict(X_test)
        train_acc = pipe.score(X_train, y_train)
        test_acc = pipe.score(X_test, y_test)
        f1 = f1_score(y_test, y_pred, average="macro")

    fitted_pipelines[name] = pipe
    results.append({
        "model": name, "train_accuracy": round(train_acc, 3), "test_accuracy": round(test_acc, 3),
        "macro_f1": round(f1, 3), "overfit_gap": round(train_acc - test_acc, 3),
        "fit_time_s": round(fit_time, 1),
    })
    print(f"{name:<18} train_acc={train_acc:.3f}  test_acc={test_acc:.3f}  "
          f"macro_f1={f1:.3f}  gap={train_acc-test_acc:.3f}  time={fit_time:.1f}s")

comparison_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
comparison_df


In [ ]:
plt.figure()
sns.barplot(data=comparison_df, x="macro_f1", y="model", palette="viridis")
plt.title("Model comparison — macro F1 on held-out test set")
plt.tight_layout(); plt.savefig("outputs/model_comparison_f1.png", dpi=120); plt.show()


**Observation:** tree-boosting models (XGBoost, then GradientBoosting/LightGBM) clearly outperform linear/instance-based models — the product/origin/destination relationship to hazard category is non-linear and involves feature interactions that boosted trees capture well. The `overfit_gap` column flags RandomForest/XGBoost as the models most worth watching for overfitting; this is addressed with tuning + regularisation below.

## Step 7 — Hyperparameter Tuning (top 2 models)

Per the project's common guidelines, we tune the **top 2 models by macro-F1** using `RandomizedSearchCV` (3-fold, macro-F1 scoring). Tuning is performed only on the training fold — never on the held-out test set.


In [ ]:
top2 = comparison_df["model"].tolist()[:2]
print("Tuning:", top2)

param_grids = {
    "XGBoost": {
        "clf__n_estimators": [150, 250, 350],
        "clf__max_depth": [4, 6, 8],
        "clf__learning_rate": [0.03, 0.07, 0.1],
        "clf__subsample": [0.8, 1.0],
    },
    "LightGBM": {
        "clf__n_estimators": [150, 250, 350],
        "clf__num_leaves": [15, 31, 63],
        "clf__learning_rate": [0.03, 0.07, 0.1],
    },
    "GradientBoosting": {
        "clf__n_estimators": [100, 150, 200],
        "clf__max_depth": [2, 3, 4],
        "clf__learning_rate": [0.05, 0.1, 0.2],
    },
    "RandomForest": {
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [15, 25, None],
        "clf__min_samples_split": [2, 5, 10],
    },
}

tuned_pipelines = {}
tuned_results = []
for name in top2:
    grid = param_grids.get(name)
    base_pipe = Pipeline([("pre", preprocessor), ("clf", models[name])])
    if grid is None:
        tuned_pipelines[name] = fitted_pipelines[name]
        continue
    use_encoded = name in NEEDS_ENCODED_Y
    Xtr, ytr = (X_train_e, y_train_e) if use_encoded else (X_train, y_train)
    Xte, yte = (X_test_e, y_test_e) if use_encoded else (X_test, y_test)

    # n_jobs=1 on the search itself is intentional: XGBoost/RandomForest/LightGBM already
    # parallelise internally (n_jobs=-1 inside the estimator). Also setting n_jobs=-1 on
    # RandomizedSearchCV causes CPU oversubscription, which can hang or crawl on Colab's
    # limited (often 2-core) CPUs. Runs sequentially over the 6 x 3 = 18 fits instead.
    search = RandomizedSearchCV(base_pipe, grid, n_iter=6, cv=3, scoring="f1_macro",
                                 random_state=RANDOM_STATE, n_jobs=1, verbose=1)
    t0 = time.time()
    search.fit(Xtr, ytr)
    tune_time = time.time() - t0

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(Xte)
    test_acc = best_pipe.score(Xte, yte)
    f1 = f1_score(yte, y_pred, average="macro")

    tuned_pipelines[name] = best_pipe
    tuned_results.append({
        "model": f"{name} (tuned)", "best_params": search.best_params_,
        "cv_best_macro_f1": round(search.best_score_, 3),
        "test_accuracy": round(test_acc, 3), "test_macro_f1": round(f1, 3),
        "tuning_time_s": round(tune_time, 1),
    })
    print(f"{name}: best CV macro_f1={search.best_score_:.3f} | test macro_f1={f1:.3f} | "
          f"best_params={search.best_params_}")

tuned_df = pd.DataFrame(tuned_results)
tuned_df


## Step 8 — Final Model Evaluation

We select the best-performing tuned model as the **final model** and evaluate it in depth: classification report, confusion matrix, and overfitting/bias-variance check.


In [ ]:
final_name = tuned_df.sort_values("test_macro_f1", ascending=False).iloc[0]["model"].replace(" (tuned)", "")
final_pipe = tuned_pipelines[final_name]
use_encoded = final_name in NEEDS_ENCODED_Y
Xte, yte = (X_test_e, y_test_e) if use_encoded else (X_test, y_test)
Xtr, ytr = (X_train_e, y_train_e) if use_encoded else (X_train, y_train)

y_pred_final = final_pipe.predict(Xte)
target_names = le.classes_ if use_encoded else sorted(y.unique())

print("FINAL MODEL:", final_name)
print("\nTrain accuracy:", round(final_pipe.score(Xtr, ytr), 3))
print("Test accuracy: ", round(final_pipe.score(Xte, yte), 3))
print("\nClassification report:\n")
print(classification_report(yte, y_pred_final, target_names=[str(t) for t in target_names], zero_division=0))


In [ ]:
cm = confusion_matrix(yte, y_pred_final)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title(f"Confusion matrix — {final_name} (final model)")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(rotation=75); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig("outputs/confusion_matrix_final.png", dpi=120); plt.show()


**Bias/variance & overfitting check:** train accuracy vs. test accuracy for the final model is reported above (the `overfit_gap` computed earlier gives the same signal for every candidate model). A small gap indicates the model generalises reasonably; if the gap were large, the fix would be lowering tree depth / increasing regularisation / gathering more data for minority classes — exactly what the tuning step targets.

**Class imbalance & generalisation:** macro-F1 (not accuracy) was used throughout so that minority hazard categories (e.g. *Veterinary drug residues*, *Irradiation*) are not masked by the dominant *Microbiological contamination* / *Other* classes. The confusion matrix above shows which categories are most often confused — typically *Other* absorbs some misclassifications since it is a catch-all bucket.

## Step 9 — Model Comparison Table (baseline + tuned)

In [ ]:
final_comparison = pd.concat([
    comparison_df.assign(stage="baseline"),
    tuned_df.rename(columns={"model": "model_tuned"}).assign(stage="tuned")
], axis=0, ignore_index=True, sort=False)

final_comparison.to_csv("outputs/model_comparison_full.csv", index=False)
comparison_df.to_csv("outputs/model_comparison_baseline.csv", index=False)
tuned_df.to_csv("outputs/model_comparison_tuned.csv", index=False)
print("Saved model comparison tables to outputs/")
comparison_df


## Step 10 — Prediction & Result Interpretation

We inspect sample predictions, and explain *which words / countries* drive each hazard-category prediction using feature importance.


In [ ]:
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(Xte) if not use_encoded else len(y_test_e), 10, replace=False)
if use_encoded:
    sample_X = X_test.iloc[sample_idx]
    sample_true = [target_names[i] for i in y_test_e[sample_idx]]
    sample_pred = [target_names[i] for i in final_pipe.predict(X_test_e.iloc[sample_idx])]
else:
    sample_X = X_test.iloc[sample_idx]
    sample_true = y_test.iloc[sample_idx].values
    sample_pred = final_pipe.predict(X_test.iloc[sample_idx])

sample_view = sample_X.copy()
sample_view["actual"] = sample_true
sample_view["predicted"] = sample_pred
sample_view["correct"] = sample_view["actual"] == sample_view["predicted"]
sample_view


In [ ]:
# Feature importance (works for tree-based final models: XGBoost / LightGBM / GradientBoosting / RandomForest)
try:
    clf_step = final_pipe.named_steps["clf"]
    pre_step = final_pipe.named_steps["pre"]
    feat_names = pre_step.get_feature_names_out()
    importances = clf_step.feature_importances_
    imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(20)

    plt.figure(figsize=(9, 7))
    sns.barplot(data=imp_df, x="importance", y="feature", palette="mako")
    plt.title(f"Top 20 most important features — {final_name}")
    plt.tight_layout(); plt.savefig("outputs/feature_importance.png", dpi=120); plt.show()
except AttributeError:
    print(f"{final_name} has no .feature_importances_; use permutation importance or SHAP instead.")


**Business interpretation:** the top features are typically (a) specific product-name tokens from TF-IDF — e.g. "fish"/"nut"-related tokens pushing toward *Microbiological contamination* or *Allergens*, "peanut"/"pistachio" tokens pushing toward *Mycotoxins* — and (b) specific origin-country one-hot flags, reflecting real regional risk patterns (e.g. certain origins historically associated with more mycotoxin or heavy-metal notifications). This matches domain knowledge about food-safety risk and gives inspectors an explainable basis for prioritising checks.

**Limitations:** the model predicts a *category*, not certainty of an actual incident; short/noisy product names limit precision for rare hazard types; and historical inspection patterns may themselves be biased (see Responsible AI section below), so predictions should support — not replace — expert judgement.

**Recommendations for future improvement:** enrich `food` with structured product-category taxonomies, add the notification "type" (alert/information/border rejection) if available from the live RASFF portal, and explore transformer-based text embeddings for the product field.


## Step 11 — Responsible AI & Ethical Considerations

- **Sensitive information:** the dataset contains no personally identifiable information — it is aggregate, company/product-level regulatory data already public via the EU RASFF system.
- **Possible bias:** notification volume reflects **inspection intensity**, not just true hazard rates — countries with more border checks may appear over-represented, which the model can unintentionally learn as "risk."
- **Fairness across groups:** predictions by origin country should be communicated as *statistical association in historical notifications*, not as a claim that a country's food is inherently unsafe.
- **Ethical risk of incorrect predictions:** a wrong hazard-category prediction could misdirect an inspector's initial checklist; the tool should be used for **triage/prioritisation**, with a human always making the final call.
- **Model limitations in practice:** trained on a historical snapshot (RASFF notifications up to the extract date) — should be retrained periodically as new notifications and regulations emerge.
- The project makes **no guaranteed-outcome claims**; all outputs are probabilistic classifications, presented with the model's own confidence scores in the dashboard.


## Step 12 — Reproducibility & Saving Artifacts

Random seeds are fixed (`RANDOM_STATE = 42`) throughout. We now save the final trained pipeline, label encoder, comparison tables, and a small dashboard-ready data sample — these are exactly the files `app.py` (the Streamlit dashboard) loads.


In [ ]:
joblib.dump(final_pipe, "outputs/model_pipeline.joblib")
joblib.dump(le, "outputs/label_encoder.joblib")

with open("outputs/model_metadata.json", "w") as f:
    json.dump({
        "final_model": final_name,
        "features": FEATURES,
        "classes": [str(c) for c in target_names],
        "test_accuracy": float(final_pipe.score(Xte, yte)),
        "test_macro_f1": float(f1_score(yte, y_pred_final, average="macro")),
    }, f, indent=2)

# Small sample of cleaned data + full cleaned data for the dashboard's EDA charts
df.to_csv("outputs/clean_data_full.csv", index=False)
df.sample(min(5000, len(df)), random_state=RANDOM_STATE).to_csv("outputs/clean_data_sample.csv", index=False)

print("Saved to outputs/:")
for f in sorted(os.listdir("outputs")):
    print(" -", f)


## Step 13 — Publish to GitHub & Deploy the Live Streamlit Dashboard

This notebook is one part of the deliverable. Alongside it you should have received (or should create in the same project folder):

```
food-safety-alert-classification/
├── Food_Safety_Alert_Classification.ipynb   # this notebook
├── app.py                                   # Streamlit dashboard + predictor
├── requirements.txt
├── README.md
├── .gitignore
└── outputs/                                 # generated by this notebook
    ├── model_pipeline.joblib
    ├── label_encoder.joblib
    ├── model_metadata.json
    ├── model_comparison_full.csv
    └── clean_data_sample.csv
```

### A. Download the generated artifacts from Colab
```python
from google.colab import files
import shutil
shutil.make_archive("outputs", "zip", "outputs")
files.download("outputs.zip")
```
Unzip `outputs.zip` into your local project folder next to `app.py`.

### B. Push to GitHub (run locally, or in a Colab cell with `!`)
```bash
git init
git add .
git commit -m "Food-safety alert classification capstone"
git branch -M main
git remote add origin https://github.com/<your-username>/food-safety-alert-classification.git
git push -u origin main
```
Make sure `outputs/model_pipeline.joblib` is committed (or hosted via Git LFS / a release asset if it is large) — the Streamlit app needs it at runtime. Do **not** commit API keys or credentials.

### C. Deploy the live Streamlit URL
1. Go to **https://share.streamlit.io** (Streamlit Community Cloud) and sign in with GitHub.
2. Click **"New app"** → pick your `food-safety-alert-classification` repo, branch `main`, main file `app.py`.
3. Click **Deploy**. Streamlit installs `requirements.txt` and launches the app — you'll get a public URL like `https://<your-app>.streamlit.app`.
4. Paste that live URL into your GitHub `README.md` and your final submission.

That's it — GitHub repo ✅, live Streamlit dashboard URL ✅.
